# 2D charge-density heatmaps

This notebook reads a downloaded square-lattice campaign and reconstructs physical `(x, y)` coordinates from the package's interleaved QTT site indexing. It is analysis-only and does not modify the solver environment.

Use the existing `Julia MPO density` Jupyter kernel installed for the 1D analysis notebook. It needs `CairoMakie` and `TOML`.

In [ ]:
using CairoMakie
using TOML
using Statistics

In [ ]:
# Change only the final directory name to view MPO results once available.
result_root = joinpath(
    ENV["HOME"], "MPO_HF_analysis", "results",
    "separable_aubry_andre_lside5_seed0p1_dense_hf",
)

In [ ]:
unquote(value::AbstractString) = strip(strip(value), '"')

function read_site_density(path::AbstractString)
    lines = readlines(path)
    strip(first(lines)) == "\"site\",\"density\"" || error("unexpected density CSV header: $path")
    sites, density = Int[], Float64[]
    for line in Iterators.drop(lines, 1)
        isempty(strip(line)) && continue
        fields = split(strip(line), ','; limit=2)
        push!(sites, parse(Int, unquote(fields[1])))
        push!(density, parse(Float64, unquote(fields[2])))
    end
    sites == collect(1:length(sites)) || error("site indices are not contiguous")
    return sites, density
end

function qtt_square_coordinate(site::Integer, total_bits::Integer)
    iseven(total_bits) || error("square lattice requires an even total bit count")
    z = site - 1
    x = y = 0
    for bit in 0:(total_bits ÷ 2 - 1)
        y |= ((z >> (2bit)) & 1) << bit
        x |= ((z >> (2bit + 1)) & 1) << bit
    end
    return x, y
end

function density_grid(sites, density)
    N = length(sites)
    total_bits = round(Int, log2(N))
    2^total_bits == N && iseven(total_bits) || error("expected N=2^L with even L, got N=$N")
    side = 2^(total_bits ÷ 2)
    grid = zeros(Float64, side, side)
    for (site, value) in zip(sites, density)
        x, y = qtt_square_coordinate(site, total_bits)
        grid[x + 1, y + 1] = value
    end
    return grid
end

function load_campaign(root::AbstractString)
    isdir(root) || error("result_root is not a directory: $root")
    tasks = sort(filter(path -> isdir(path) && isfile(joinpath(path, "site_density.csv")), readdir(root; join=true)))
    isempty(tasks) && error("no completed task directories below $root")
    [begin
        sites, density = read_site_density(joinpath(task, "site_density.csv"))
        (label=replace(basename(task), r"^task_\d+_" => ""), density=density, grid=density_grid(sites, density), observables=TOML.parsefile(joinpath(task, "observables.toml")))
    end for task in tasks]
end

In [ ]:
cases = load_campaign(result_root)
[(case=entry.label, N=length(entry.density), density_min=minimum(entry.density), density_max=maximum(entry.density), particle_number=entry.observables["particle_number"], energy_total=entry.observables["energy_total"]) for entry in cases]

In [ ]:
# A common colour scale makes density contrast comparable across cases.
all_density = reduce(vcat, (entry.density for entry in cases))
contrast = maximum(abs.(all_density .- 0.5))
colour_range = (0.5 - contrast, 0.5 + contrast)

for entry in cases
    side = size(entry.grid, 1)
    fig = Figure(size=(760, 680))
    axis = Axis(fig[1, 1], xlabel="x", ylabel="y", aspect=DataAspect(),
        title="Charge density — $(entry.label)")
    map = heatmap!(axis, 0:(side - 1), 0:(side - 1), entry.grid;
        colormap=:balance, colorrange=colour_range)
    Colorbar(fig[1, 2], map, label="density n(x,y)")
    display(fig)
end

For the MPO campaign, change the final component of `result_root` to `separable_aubry_andre_lside5_seed0p1`. The two result trees have matching task labels and can then be compared pointwise.